In [27]:
%load_ext autoreload
%autoreload 2
   
import pandas as pd
from datetime import datetime
from utils.preprocessing import (
    load_raw_reviews,
    filter_negative_ratings,
    add_relative_time_columns,
    drop_exact_duplicates,
    normalize_text,
    apply_normalization,
    load_slang_dict,
    normalize_slang,
    apply_slang_normalization,
    filter_short_reviews,
    save_preprocessed_outputs,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# Tahap 1: Load raw reviews & filter rating 1-2
df_wondr = load_raw_reviews("data/raw/wondr_by_BNI_raw.csv")
df_wondr_neg = filter_negative_ratings(df_wondr)
df_wondr_neg.head()

✅ Loaded 44,627 reviews from data/raw/wondr_by_BNI_raw.csv
✅ Rating filter: kept 10,319 / 44,627 reviews (dropped 34,308 non-negative reviews).
   Distribution: {1: 8313, 2: 2006}


,review_id,app_id,app_name,user_name_hash,rating,review_text,thumbs_up_count,review_created_version,reply_content,reply_date,date_utc,date_wib,date_local,word_count,char_count,scrape_timestamp
2,79760a72-d072-4274-b151-b380ef996bf8,id.bni.wondr,wondr by BNI,c82060bf881261c1,1,bagus banget,0,1.4.0,"Hai Kak Cv Nam Kupang, makasih ya udah pakai w...",2025-07-04T16:50:17+00:00,2025-07-04T16:45:55+00:00,2025-07-04 23:45:55,2025-07-04,2,12,2026-04-20T11:00:19.325849+07:00
14,a88d3ee3-ccb9-4376-8c26-e83d0a599672,id.bni.wondr,wondr by BNI,7a013a4fcebabe05,1,"verifikasi sulit untuk hp lama,. sampe geleng""...",0,1.3.2,"Halo Kak Nico Prasetya, maaf atas kendalanya. ...",2025-07-04T16:05:00+00:00,2025-07-04T15:58:26+00:00,2025-07-04 22:58:26,2025-07-04,12,73,2026-04-20T11:00:19.326851+07:00
18,3fce2166-f6b0-4a6f-807d-4c2a7671ae14,id.bni.wondr,wondr by BNI,261f843ad1b035f6,1,APAAN INI APK WONDER MASAK SISA 52 RIBU NARIK ...,0,1.4.0,"Halo Kak Apriansen Saragi, maaf atas kendalany...",2025-07-04T16:04:08+00:00,2025-07-04T15:35:44+00:00,2025-07-04 22:35:44,2025-07-04,30,152,2026-04-20T11:00:19.326851+07:00
27,72028fd7-3f8f-4ad4-ae60-8579e1ac35fc,id.bni.wondr,wondr by BNI,de30c69d5c893b10,1,ribet banget diverifikasi wajah,0,1.4.0,"Hai Kak Matahari Pagi, maaf utk kendalanya. Pa...",2025-07-04T14:48:47+00:00,2025-07-04T14:37:24+00:00,2025-07-04 21:37:24,2025-07-04,4,31,2026-04-20T11:00:19.326851+07:00
46,f9ad7b8c-c3d3-4043-8621-257b6c8e6a6b,id.bni.wondr,wondr by BNI,d9fc03eba6985652,1,terlalu banyak perbaikan tidak nyaman saat tra...,0,1.4.0,"Halo Kak Bayu Sateng, maaf atas kendalanya. Sa...",2025-07-04T13:42:37+00:00,2025-07-04T13:17:23+00:00,2025-07-04 20:17:23,2025-07-04,8,60,2026-04-20T11:00:19.327848+07:00


In [3]:
# Tahap 2: Ekstraksi relative_month & relative_week
wondr_launch = datetime(2024, 7, 5)
df_wondr_temporal = add_relative_time_columns(df_wondr_neg, wondr_launch)
df_wondr_temporal[["date_wib", "relative_month", "relative_week", "rating"]].head(10)

✅ Relative time extraction (launch=2024-07-05, window=12 months):
   Kept 10,319 / 10,319 reviews in window.
   Distribution per relative_month: {1: 327, 2: 431, 3: 595, 4: 769, 5: 3005, 6: 1487, 7: 809, 8: 713, 9: 552, 10: 439, 11: 457, 12: 735}


,date_wib,relative_month,relative_week,rating
2,2025-07-04 23:45:55,12,52,1
14,2025-07-04 22:58:26,12,52,1
18,2025-07-04 22:35:44,12,52,1
27,2025-07-04 21:37:24,12,52,1
46,2025-07-04 20:17:23,12,52,1
56,2025-07-04 19:36:17,12,52,1
70,2025-07-04 18:33:29,12,52,1
85,2025-07-04 17:33:03,12,52,1
90,2025-07-04 17:13:42,12,52,1
115,2025-07-04 15:48:01,12,52,2


In [4]:
# Tahap 3: Hapus exact duplicates
df_wondr_dedup = drop_exact_duplicates(df_wondr_temporal)

✅ Exact duplicate removal (by 'review_text', keep first):
   Kept 10,131 / 10,319 reviews (dropped 188 duplicates).

   Top 5 most-duplicated review texts:
     1. [12x] 'mantap'
     2. [12x] 'Sering error'
     3. [11x] 'Ribet'
     4. [11x] 'baik'
     5. [11x] 'ok'


In [5]:
# Tahap 4: Apply text normalization ke seluruh data
df_wondr_clean = apply_normalization(df_wondr_dedup)

# Inspeksi visual: sample sebelum & sesudah
df_wondr_clean[["review_text", "review_text_cleaned", "word_count_after"]].sample(10, random_state=42)

✅ Text normalization applied to 10,131 reviews.
   Empty after cleaning: 4 reviews (will be dropped at Stage 7 short-review filter).
   Word count (after) — mean: 17.7, median: 13, max: 91


,review_text,review_text_cleaned,word_count_after
3780,mau buat transaksi gak bisa . kecewa banget pa...,mau buat transaksi gak bisa kecewa banget pake...,30
532,Susah bgt verifikasi wajah ribet asli,susah bgt verifikasi wajah ribet asli,6
1155,Halah.. sama saja pun.. malah lebih lengkap bn...,halah sama saja pun malah lebih lengkap bni mo...,45
1953,hapus saja gk guna. ribet pake foto wajah sega...,hapus saja gk guna ribet pake foto wajah segal...,21
4924,"Ribet, mau ngecek riwayat ae susah, enak BNI B...",ribet mau ngecek riwayat ae susah enak bni ban...,21
4238,hari ini ada apa aplikasi wondr.... ko tidak b...,hari ini ada apa aplikasi wondr ko tidak bisa ...,10
3038,Aplikasinya eror mulu ga pernah perjalan normal,aplikasinya eror mulu ga pernah perjalan normal,7
1441,Aplikasi Ampas apa apan di pixel 7 gabisa veri...,aplikasi ampas apa apan di pixel gabisa verif ...,18
1373,aplikasi paling jelek yg pernah ada daftar nya...,aplikasi paling jelek yg pernah ada daftar nya...,18
6870,Ini banyak fitur yg gk ada seperti tarik tunai...,ini banyak fitur yg gk ada seperti tarik tunai...,23


In [6]:
import pandas as pd

# Inspect structure
salsabila_raw = pd.read_csv("dictionaries/salsabila.csv")
print(f"Shape: {salsabila_raw.shape}")
print(f"Columns: {salsabila_raw.columns.tolist()}")
print(f"\nHead:")
salsabila_raw.head(10)

Shape: (15006, 7)
Columns: ['slang', 'formal', 'In-dictionary', 'context', 'category1', 'category2', 'category3']

Head:


,slang,formal,In-dictionary,context,category1,category2,category3
0,woww,wow,1,wow,elongasi,0,0
1,aminn,amin,1,Selamat ulang tahun kakak tulus semoga panjang...,elongasi,0,0
2,met,selamat,1,Met hari netaas kak!? Wish you all the best @t...,abreviasi,0,0
3,netaas,menetas,1,Met hari netaas kak!? Wish you all the best @t...,afiksasi,elongasi,0
4,keberpa,keberapa,0,Birthday yg keberpa kak?,abreviasi,0,0
5,eeeehhhh,eh,1,Eh ada @ghessawarsana . Eeeehhhh,elongasi,0,0
6,kata2nyaaa,kata-katanya,0,Kata2nyaaa ?,reduplikasi,elongasi,0
7,hallo,halo,1,Hallo kakak tulus,elongasi,0,0
8,kaka,kakak,1,Ngefans banget sama kaka? sukses selalu ka? @t...,zeroisasi,0,0
9,ka,kak,1,Ngefans banget sama kaka? sukses selalu ka? @t...,zeroisasi,0,0


In [7]:
# Indonesian standard words to protect from mis-replacement.
# Identified via data-driven inspection of most-triggered Salsabila mappings.
INDONESIAN_PROTECTED_WORDS = {
    'apa',      # Salsabila: 'apaa' → 'diapa' (after preprocess: 'apa' → 'diapa')
    'nya',      # Salsabila: 'nyaa' → 'dua-duanya' → catastrophic
    'sekali',   # Salsabila: 'sekalii' → 'sekali kali' (semantic shift)
    'bener',    # Salsabila: 'benerr' → 'benar benar' (mis-reduplikasi)
    'minta',    # Salsabila: 'mintaa' → 'meminta' (POS shift verb)
    'bukan',    # Salsabila: 'bukann' → 'bukannya' (semantic shift negation)
}

slang_dict = load_slang_dict(
    salsabila_path="dictionaries/salsabila.csv",
    banking_ext_path="dictionaries/banking_extension.csv",
    blocklist=INDONESIAN_PROTECTED_WORDS,
)

   Dropped 348 degrading reduplication mappings (e.g., 'baru' → 'baru baru').
   Dropped 71 blocklisted entries (Indonesian standard words protected from mis-replacement).
✅ Loaded Salsabila lexicon: 3,641 entries (after preprocessing & dedup).
✅ Loaded banking extension: 39 entries.
✅ Merged dictionary: 3,675 total entries (5 banking entries override Salsabila).


In [8]:
for word in INDONESIAN_PROTECTED_WORDS:
    print(f"'{word}' →", repr(slang_dict.get(word, 'NOT IN DICT')))

'bukan' → 'NOT IN DICT'
'sekali' → 'NOT IN DICT'
'nya' → 'NOT IN DICT'
'apa' → 'NOT IN DICT'
'bener' → 'NOT IN DICT'
'minta' → 'NOT IN DICT'


In [9]:
test_cases = [
    "aplikasinya gabisa login bgt",
    "trf gagal terus sm cs susah dihubungi",
    "verif wajah gak bisa pdhl udah coba berkali kali",
    "saldo udh masuk tp notifnya gak ada",
    "register baru tdk bisa otp lambat",
]

for original in test_cases:
    cleaned = normalize_slang(original, slang_dict)
    print(f"BEFORE: {original}")
    print(f"AFTER : {cleaned}")
    print("-" * 70)

BEFORE: aplikasinya gabisa login bgt
AFTER : aplikasi enggak bisa login banget
----------------------------------------------------------------------
BEFORE: trf gagal terus sm cs susah dihubungi
AFTER : transfer gagal terus sama customer service susah dihubungi
----------------------------------------------------------------------
BEFORE: verif wajah gak bisa pdhl udah coba berkali kali
AFTER : verifikasi wajah enggak bisa padahal sudah coba berkali kali
----------------------------------------------------------------------
BEFORE: saldo udh masuk tp notifnya gak ada
AFTER : saldo sudah masuk tapi notifikasi enggak ada
----------------------------------------------------------------------
BEFORE: register baru tdk bisa otp lambat
AFTER : daftar baru tidak bisa kode otp lambat
----------------------------------------------------------------------


In [10]:
print("'baru'   →", repr(slang_dict.get('baru', 'NOT IN DICT')))
print("'ada'    →", repr(slang_dict.get('ada', 'NOT IN DICT')))
print("'senang' →", repr(slang_dict.get('senang', 'NOT IN DICT')))

'baru'   → 'NOT IN DICT'
'ada'    → 'NOT IN DICT'
'senang' → 'NOT IN DICT'


In [11]:
df_wondr_slang = apply_slang_normalization(df_wondr_clean, slang_dict)
df_wondr_slang[["review_text", "review_text_cleaned", "word_count_after"]].sample(10, random_state=42)

✅ Slang normalization applied to 10,131 reviews.
   Reviews with at least one slang replacement: 7,423 (73.3%).
   Word count (after) — mean: 17.9, median: 14, max: 91


,review_text,review_text_cleaned,word_count_after
3780,mau buat transaksi gak bisa . kecewa banget pa...,mau buat transaksi enggak bisa kecewa banget p...,30
532,Susah bgt verifikasi wajah ribet asli,susah banget verifikasi wajah ribet asli,6
1155,Halah.. sama saja pun.. malah lebih lengkap bn...,halah sama saja pun malah lebih lengkap bni mo...,45
1953,hapus saja gk guna. ribet pake foto wajah sega...,hapus saja enggak guna ribet pakai foto wajah ...,22
4924,"Ribet, mau ngecek riwayat ae susah, enak BNI B...",ribet mau cek riwayat saja susah enak bni bank...,21
4238,hari ini ada apa aplikasi wondr.... ko tidak b...,hari ini ada apa aplikasi wondr kok tidak bisa...,10
3038,Aplikasinya eror mulu ga pernah perjalan normal,aplikasi eror mulu enggak pernah perjalan normal,7
1441,Aplikasi Ampas apa apan di pixel 7 gabisa veri...,aplikasi ampas apa apan di pixel enggak bisa v...,19
1373,aplikasi paling jelek yg pernah ada daftar nya...,aplikasi paling jelek yang pernah ada daftar n...,18
6870,Ini banyak fitur yg gk ada seperti tarik tunai...,ini banyak fitur yang enggak ada seperti tarik...,23


In [12]:
print("'apa' →", repr(slang_dict.get('apa', 'NOT IN DICT')))

'apa' → 'NOT IN DICT'


In [13]:
# Cari semua entry yang slang-nya pendek (<= 4 huruf) dan formal-nya berbeda
# Ini kandidat kata baku yang ke-corrupt
suspect = {k: v for k, v in slang_dict.items() 
              if len(k) <= 4 and k != v}
print(f"Total suspect entries (slang ≤ 4 chars): {len(suspect)}")
print("\nSample 30:")
for k, v in list(suspect.items())[:30]:
    print(f"  '{k}' → '{v}'")

Total suspect entries (slang ≤ 4 chars): 1121

Sample 30:
  'hr' → 'hari'
  'woww' → 'wow'
  'met' → 'selamat'
  'wktu' → 'waktu'
  'eehh' → 'eh'
  'seh' → 'sih'
  'jh' → 'saja'
  'pgn' → 'pengin'
  'sich' → 'sih'
  'kl' → 'kalo'
  'gw' → 'gue'
  'gile' → 'gila'
  'masi' → 'masih'
  'blln' → 'bulan'
  'blom' → 'belum'
  'dr' → 'dari'
  'aq' → 'aku'
  'gtu' → 'begitu'
  'yah' → 'ya'
  'akau' → 'aku'
  'rb' → 'ribu'
  'aja' → 'saja'
  'gitu' → 'begitu'
  'udah' → 'sudah'
  'yaa' → 'ya'
  'jgn' → 'jangan'
  'akuu' → 'aku'
  'ngga' → 'enggak'
  'yg' → 'yang'
  'lg' → 'lagi'


In [14]:
# Cari mappings yang ke-trigger di data, untuk identify yang corrupt
# Compare token-level: mana yang berubah?
from collections import Counter

corrupt_replacements = Counter()
for original, cleaned in zip(df_wondr_clean["review_text_cleaned"], df_wondr_slang["review_text_cleaned"]):
    orig_tokens = original.split()
    clean_tokens = cleaned.split()
    
    # Skip kalau token count sama (no expansion)
    # Token-level diff susah karena multi-word expansion bisa shift index
    # Pakai approach lain: check setiap token original, kalau ada di dict, count
    for tok in orig_tokens:
        if tok in slang_dict:
            corrupt_replacements[(tok, slang_dict[tok])] += 1

print("Top 30 most-triggered slang replacements:")
for (slang, formal), count in corrupt_replacements.most_common(30):
    print(f"  [{count}x] '{slang}' → '{formal}'")

Top 30 most-triggered slang replacements:
  [2780x] 'bni' → 'bni'
  [1611x] 'gak' → 'enggak'
  [1554x] 'ga' → 'enggak'
  [1398x] 'yg' → 'yang'
  [1225x] 'aja' → 'saja'
  [1011x] 'udah' → 'sudah'
  [994x] 'apk' → 'aplikasi'
  [693x] 'pake' → 'pakai'
  [534x] 'gk' → 'enggak'
  [505x] 'aplikasinya' → 'aplikasi'
  [322x] 'udh' → 'sudah'
  [319x] 'gabisa' → 'enggak bisa'
  [305x] 'gimana' → 'bagaimana'
  [276x] 'otp' → 'kode otp'
  [256x] 'tf' → 'transfer'
  [255x] 'mbanking' → 'mobile banking'
  [250x] 'app' → 'aplikasi'
  [228x] 'klo' → 'kalo'
  [221x] 'tp' → 'tapi'
  [220x] 'bgt' → 'banget'
  [220x] 'sampe' → 'sampai'
  [212x] 'trus' → 'terus'
  [201x] 'gini' → 'begini'
  [200x] 'tdk' → 'tidak'
  [176x] 'ni' → 'ini'
  [172x] 'sdh' → 'sudah'
  [172x] 'dah' → 'deh'
  [166x] 'suka' → 'sukanya'
  [158x] 'in' → 'ini'
  [153x] 'kaya' → 'kayak'


In [15]:
# Row 4238 dan 1441 sebelumnya bermasalah
problematic_indices = [4238, 1441]
df_wondr_slang.loc[problematic_indices, ["review_text", "review_text_cleaned"]]

,review_text,review_text_cleaned
4238,hari ini ada apa aplikasi wondr.... ko tidak b...,hari ini ada apa aplikasi wondr kok tidak bisa...
1441,Aplikasi Ampas apa apan di pixel 7 gabisa veri...,aplikasi ampas apa apan di pixel enggak bisa v...


In [16]:
from utils.preprocessing import INDONESIAN_PROTECTED_WORDS
print(INDONESIAN_PROTECTED_WORDS)

{'bukan', 'apa', 'nya', 'sekali', 'bener', 'minta'}


In [21]:
# Tahap 6: Filter short reviews (<5 kata)
df_wondr_short = filter_short_reviews(df_wondr_slang, min_words=5)

✅ Short review filter (min_words=5):
   Kept 8,982 / 10,131 reviews (dropped 1,149 reviews with < 5 words).
   Word count (after) — mean: 19.8, median: 15, max: 91


In [22]:
# Load model (5-10 detik di first call)
lang_model = load_lang_detector("lid.176.bin")

# Test dengan beberapa text variasi
from utils.preprocessing import detect_language_tier

test_texts = [
    "aplikasi tidak bisa login terus error",       # confident Indonesian
    "the app is broken cannot login",              # confident English
    "login gabisa app error padahal sudah update", # code-switching id+en
    "app error",                                   # too short, ambiguous
    "aplikasi bagus sekali mantap",                # short Indonesian
]

for t in test_texts:
    result = detect_language_tier(t, lang_model)
    print(f"  {result} | '{t}'")

✅ Loaded fasttext language detector from lid.176.bin
  ('id', 0.9371704459190369, 1) | 'aplikasi tidak bisa login terus error'
  ('en', 0.9359691739082336, 3) | 'the app is broken cannot login'
  ('id', 0.5028926730155945, 1) | 'login gabisa app error padahal sudah update'
  ('en', 0.2530179023742676, 3) | 'app error'
  ('id', 0.4869161546230316, 2) | 'aplikasi bagus sekali mantap'


In [13]:
import fasttext
import numpy as np

model = fasttext.load_model("lid.176.bin")
result = model.f.predict("aplikasi tidak bisa login", 3, 0.0, "strict")
print("Type:", type(result))
print("Length:", len(result))
print("Items:")
for i, item in enumerate(result):
    print(f"  [{i}] type={type(item).__name__}, value={item}")

Type: <class 'list'>
Length: 3
Items:
  [0] type=tuple, value=(0.9876810908317566, '__label__id')
  [1] type=tuple, value=(0.003023119643330574, '__label__eu')
  [2] type=tuple, value=(0.0026576598174870014, '__label__pl')


In [23]:
# Tahap 6a: Detect language untuk semua reviews
df_wondr_lang = apply_language_detection(df_wondr_short, lang_model)

✅ Language detection applied to 8,982 reviews.
   Tier distribution:
     Tier 1 (KEEP confident-id): 8,133 (90.5%)
     Tier 2 (KEEP flagged): 726 (8.1%)
     Tier 3 (DROP non-id): 123 (1.4%)
   Top 5 non-Indonesian languages detected:
     'ms': 168
     'en': 47
     'it': 13
     'pl': 8
     'hu': 6


In [24]:
# Sample dari setiap tier untuk visual check
for tier in [1, 2, 3]:
    print(f"\n{'='*70}")
    print(f"TIER {tier} — Sample 5 reviews")
    print('='*70)
    sample = df_wondr_lang[df_wondr_lang["lang_tier"] == tier].sample(
        min(5, (df_wondr_lang["lang_tier"] == tier).sum()),
        random_state=42
    )
    for _, row in sample.iterrows():
        print(f"  [{row['lang_top']} {row['lang_top_conf']:.2f}] '{row['review_text_cleaned'][:80]}'")


TIER 1 — Sample 5 reviews
  [id 0.96] 'aplikasi hari tidak bisa digunakan'
  [id 0.95] 'baru juga tadi ng download ini di suruh di tempat kerja tapi enggak bisa verivik'
  [id 0.62] 'login lama padahal jaringan bagus buka aplikasi bank lain malah lancar tanpa ken'
  [id 0.93] 'semenjak pakai wondr by bni kalau mau mengirim sudah tidak bisa kosong padahal s'
  [id 0.79] 'enggak jelas banget mau login saja susah bolak balik ke menu syarat dan pengguna'

TIER 2 — Sample 5 reviews
  [id 0.33] 'wondr by bni kenapa leleet bahkan gagal masuk'
  [id 0.37] 'kacau banget tak ada kemajuan'
  [id 0.23] 'kenapa top up gopay gangguan terus'
  [id 0.44] 'ribet sekali lupa password malah disuruh scan ulang ktp'
  [id 0.34] 'sering gagal transfer di virtual account padahal sisa saldo masih mencukupi'

TIER 3 — Sample 5 reviews
  [jv 0.13] 'tolong kalo maintenance jangan di jam sibuk dong'
  [en 0.09] 'eror mulu bni wondr parah'
  [ms 0.50] 'payah lah gangguan mulu wndr bni'
  [es 0.10] 'repot tak ada 

In [25]:
# Audit Tier 3 sekarang — apakah false negative berkurang?
tier3_now = df_wondr_lang[df_wondr_lang["lang_tier"] == 3]
print(f"\nTier 3 count: {len(tier3_now)}")
print("\nSample 10 Tier 3 (sekarang):")
sample = tier3_now.sample(min(10, len(tier3_now)), random_state=42)
for _, row in sample.iterrows():
    print(f"  [{row['lang_top']} {row['lang_top_conf']:.2f}] '{row['review_text_cleaned'][:80]}'")


Tier 3 count: 123

Sample 10 Tier 3 (sekarang):
  [jv 0.13] 'tolong kalo maintenance jangan di jam sibuk dong'
  [en 0.09] 'eror mulu bni wondr parah'
  [ms 0.50] 'payah lah gangguan mulu wndr bni'
  [es 0.10] 'repot tak ada setor tarik non kartu'
  [fr 0.19] 'saldo kok terpotong tiba anjk'
  [en 0.17] 'maintenance terus bikin ribet kalo mau transaksi'
  [ms 0.47] 'minta kode otp saja susah amet'
  [ms 0.36] 'gangguan mulu perbaikan mulu bikin enggak nyaman'
  [ms 0.22] 'sering banget gangguan jelek banget'
  [ms 0.24] 'tarik tunai tanpa kartu pulsa kepotong enggak kayak bca'


In [26]:
# Berapa review di Tier 3 yang punya top_conf < 0.5?
# Itu yang akan ke-rescue dengan Opsi C
tier3 = df_wondr_lang[df_wondr_lang["lang_tier"] == 3]
n_low_conf = (tier3["lang_top_conf"] < 0.5).sum()
n_high_conf = (tier3["lang_top_conf"] >= 0.5).sum()

print(f"Tier 3 total: {len(tier3)}")
print(f"  - Low conf (<0.5): {n_low_conf} → would be rescued to Tier 2")
print(f"  - High conf (>=0.5): {n_high_conf} → still Tier 3")

print("\nSample Tier 3 dengan high conf (>=0.5) — these are confident non-Indonesian:")
high_conf_sample = tier3[tier3["lang_top_conf"] >= 0.5].sample(
    min(10, n_high_conf), random_state=42
)
for _, row in high_conf_sample.iterrows():
    print(f"  [{row['lang_top']} {row['lang_top_conf']:.2f}] '{row['review_text_cleaned'][:80]}'")

Tier 3 total: 123
  - Low conf (<0.5): 103 → would be rescued to Tier 2
  - High conf (>=0.5): 20 → still Tier 3

Sample Tier 3 dengan high conf (>=0.5) — these are confident non-Indonesian:
  [en 0.70] 'mash belum bisa terima dana by qr'
  [en 0.61] 'live agent customer service slow respon'
  [en 0.50] 'mending mobile banking wondr sering eror'
  [ms 0.83] 'lebih baik bni mobile bank daripada ini'
  [ms 0.70] 'ini kalo maintenance kenpa pas dijam kerja kenpa enggak malam saja'
  [fi 0.52] 'kenapa transsaksi gagal tapi limit terpotong'
  [sw 0.57] 'siap siap potong gaji ya bang dev'
  [ms 0.56] 'kamera jernih tempat terang kepala kanan kiri bawa gagal terus kacau'
  [ms 0.58] 'keseringan ganguan dalam berteransaksi jelek sekali lah ganguan teros'
  [ms 0.53] 'ubur ubur ikan lele dikit dikit eror boleh'


## Audit Conclusion: Language Filter Deprecated

Berdasarkan audit di atas:
- Tier 3 total: 123 reviews (1.4%)
- 84% punya low confidence (<0.5) → uncertain predictions
- 9/10 high-confidence Tier 3 samples adalah FALSE POSITIVE (valid Indonesia)

**Decision:** Language filter (fasttext lid.176) tidak digunakan di pipeline final
karena terbukti punya systemic bias di domain banking Indonesia. Detail di
preprocessing.py (Stage 7 DEPRECATED).

Audit ini disimpan untuk dokumentasi metodologis (Bab 3 skripsi).

In [28]:
# Tahap 8: Save 2 versi (BERTopic-ready + full audit)
save_preprocessed_outputs(
    df_wondr_short,
    bertopic_path="data/processed/wondr_bertopic.csv",
    full_path="data/processed/wondr_full.csv",
)

✅ BERTopic-ready saved: data/processed/wondr_bertopic.csv
   Shape: (8982, 6), Columns: ['review_id', 'review_text_cleaned', 'relative_month', 'relative_week', 'date_wib', 'rating']

✅ Full audit saved: data/processed/wondr_full.csv
   Shape: (8982, 20), Columns: 20 columns


In [29]:
# Validasi 1: Distribusi review per relative_month
print("Distribusi review per relative_month:")
print(df_wondr_short["relative_month"].value_counts().sort_index().to_string())
print(f"\nTotal final reviews: {len(df_wondr_short):,}")
print(f"Mean per month: {len(df_wondr_short) / 12:.0f}")

Distribusi review per relative_month:
relative_month
1      302
2      395
3      544
4      678
5     2568
6     1297
7      727
8      621
9      462
10     357
11     388
12     643

Total final reviews: 8,982
Mean per month: 748


In [30]:
# Validasi 2: Sample 30 review untuk visual check
sample = df_wondr_short.sample(30, random_state=42)
for _, row in sample.iterrows():
    print(f"[M{row['relative_month']:02d} R{row['rating']}] {row['review_text_cleaned'][:100]}")

[M09 R1] makin susah dibuka sekarang tolong perbaiki aplikasi ini min
[M01 R1] baru install wondr sempat berhasil login tapi lupa password setelah itu verifikasi ktp tapi selalu g
[M03 R1] susah banget pas vermuknya sudah berkali coba tetap gagal mulu woi
[M12 R2] katanya kartu atm bakal dikirim dirumah tapi nyatanya enggak sudah berbulan enggak dikirim dikirim p
[M12 R1] sayang saldo mengendap yang harus disisakan cukup besar sebesar ribu terima kasih atas pelayan yang 
[M09 R1] kalau sudah jam kenapa enggak bisa transfer ris dan lain kalau begini terusending caring bank lain c
[M05 R1] habis update kok enggak bisa dipakai aplikasi
[M08 R1] saat transaksi masih proses dan saldo terpotong sangat lama untuk memeriksa nya harus menunggu hk
[M09 R1] topup dana enggak masuk woy
[M03 R1] aplikasi enggak bisa di pakai tolong diperbaiki buat kelancaran nasabah masak mau masuk saja ribet b
[M07 R1] kenapa pas verifikasi wajah gagal mulu padahal pencahayaan bagus sudah mengikuti petunjuk tapi t

In [31]:
# Validasi 3: Top 100 most frequent words
from collections import Counter

all_words = " ".join(df_wondr_short["review_text_cleaned"].fillna("")).split()
word_counts = Counter(all_words)

print(f"Total unique words: {len(word_counts):,}")
print(f"Total tokens: {sum(word_counts.values()):,}")
print(f"\nTop 100 most frequent words:")
for i, (word, count) in enumerate(word_counts.most_common(100), 1):
    print(f"  {i:3d}. {word:25s} ({count:,})")

Total unique words: 8,026
Total tokens: 177,685

Top 100 most frequent words:
    1. aplikasi                  (5,361)
    2. enggak                    (4,674)
    3. bisa                      (4,102)
    4. di                        (3,727)
    5. sudah                     (2,877)
    6. ini                       (2,856)
    7. bni                       (2,765)
    8. yang                      (2,491)
    9. tidak                     (2,353)
   10. saya                      (2,227)
   11. nya                       (2,034)
   12. ada                       (1,897)
   13. mau                       (1,868)
   14. lagi                      (1,824)
   15. saja                      (1,713)
   16. dan                       (1,679)
   17. terus                     (1,595)
   18. transaksi                 (1,448)
   19. verifikasi                (1,436)
   20. mobile                    (1,427)
   21. pakai                     (1,416)
   22. tapi                      (1,403)
   23. ke           